# 00 — Setup & Configuration

Single-agent + file-based Agent Skill alternative to the multi-agent Impossible Travel Analyzer.

Run this once to configure your Microsoft Foundry connection and verify connectivity. 
Configuration is saved to `workshop_config.json` so `01-investigation.ipynb` and `02-deploy.ipynb` can load it automatically.

### Prerequisites
- Python 3.13+
- Azure CLI installed and logged in (`az login`)
- A Microsoft Foundry project with a deployed model (e.g., `gpt-4o`, `gpt-4.1-mini`)


In [ ]:
# Install dependencies
%pip install -q -r requirements.txt


In [1]:
# Foundry Connection — edit the 4 variables, or rely on a previously-saved workshop_config.json.
import json
from pathlib import Path

# ──────────────────────────────────────────────────────────────
# ✏️  Edit these to connect to your Foundry resource
# ──────────────────────────────────────────────────────────────
RESOURCE_GROUP        = "rg-foundry-iac-ops"
ACCOUNT_NAME          = "fndryiac2ttkx3"
PROJECT_NAME          = "iac-ops-demo"
MODEL_DEPLOYMENT_NAME = "gpt-4.1-mini"
# ──────────────────────────────────────────────────────────────

config_path = Path("workshop_config.json")
if config_path.exists() and not ACCOUNT_NAME:
    with open(config_path) as f:
        config = json.load(f)
    RESOURCE_GROUP        = config.get("RESOURCE_GROUP", RESOURCE_GROUP)
    ACCOUNT_NAME          = config.get("ACCOUNT_NAME", ACCOUNT_NAME)
    PROJECT_NAME          = config.get("PROJECT_NAME", PROJECT_NAME)
    MODEL_DEPLOYMENT_NAME = config.get("MODEL_DEPLOYMENT_NAME", MODEL_DEPLOYMENT_NAME)
    print(f"✅ Loaded saved config from {config_path}")

if not ACCOUNT_NAME:
    raise ValueError("Edit RESOURCE_GROUP / ACCOUNT_NAME / PROJECT_NAME / MODEL_DEPLOYMENT_NAME above.")

PROJECT_ENDPOINT = f"https://{ACCOUNT_NAME}.services.ai.azure.com/api/projects/{PROJECT_NAME}"
print(f"Resource Group:      {RESOURCE_GROUP}")
print(f"Account Name:        {ACCOUNT_NAME}")
print(f"Project Name:        {PROJECT_NAME}")
print(f"Model Deployment:    {MODEL_DEPLOYMENT_NAME}")
print(f"Project Endpoint:    {PROJECT_ENDPOINT}")


Resource Group:      rg-foundry-iac-ops
Account Name:        fndryiac2ttkx3
Project Name:        iac-ops-demo
Model Deployment:    gpt-4.1-mini
Project Endpoint:    https://fndryiac2ttkx3.services.ai.azure.com/api/projects/iac-ops-demo


In [2]:
# Verify Foundry connectivity
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

credential = AzureCliCredential()
af_client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT_NAME,
    credential=credential,
)

_test_agent = Agent(client=af_client, instructions="Reply with only the word 'OK'.", name="ConnTest")
_test_resp = await _test_agent.run("ping")
print(f"✅ Foundry connection verified: {_test_resp.text[:50]}")
del _test_agent, _test_resp


<frozen abc>:106: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
<frozen abc>:106: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.


✅ Foundry connection verified: OK


In [3]:
# Save config for the other notebooks
config = {
    "RESOURCE_GROUP":        RESOURCE_GROUP,
    "ACCOUNT_NAME":          ACCOUNT_NAME,
    "PROJECT_NAME":          PROJECT_NAME,
    "MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT_NAME,
    "PROJECT_ENDPOINT":      PROJECT_ENDPOINT,
}
with open("workshop_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("✅ Config saved. Run 01-investigation.ipynb next.")


✅ Config saved. Run 01-investigation.ipynb next.
